# 11 · 80-slice Raptor corpus, shard 2/3 (Kaggle, CPU)

**Why?** The Raptor arms trained on the 44-slice public corpus (07) looked good on the gold studies but lowered the
public blend on the leaderboard (0.956 → 0.953). The public blend's models were trained with denser slice sampling
(80–96 slices) and score ~0.94 alone. By the Raptor author's measurements slice density is the strongest lever
(44 → 80 slices: 0.926 → 0.932; ~0.939 with the full recipe).

**Geometry ("Fine Spacing", the author's recipe in discussion 737696):** 80 slices, slots 22 sagittal-fluid /
18 sagittal / 15 coronal-fluid / 10 coronal / 15 axial, slice span 2–98% of each series, 140 mm centre crop, 336 px,
series-level 2–98% intensity window.

**Critical:** the corpus is built with **the same** `build_study` function used at prediction time
(`third_party/raptor/infer_knee.py`), so training and test geometry are identical.

**Size:** 4,407 × 80 × 336 × 336 ≈ 40 GB → 3 shards (~13 GB) because of Kaggle's 20 GB output limit. The training
script joins the shards in order from their `corpus_vols_sNN.npy` names.

In [ ]:
# Add the project root (the folder that contains src/) to the Python path; works locally and on Kaggle.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk with pruning: never enters the competition DICOM folders (rglob took minutes)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ not found")

PROJECT_ROOT = add_project_root()
print("Project root:", PROJECT_ROOT)

In [ ]:
import importlib.util, time
from multiprocessing import Pool
from pathlib import Path
import numpy as np, pandas as pd
from src.config import DATA_DIR, IS_KAGGLE, find_input

SHARD, N_SHARDS = 1, 3
SLOTS80 = [("Sagittal", 1, 22), ("Sagittal", 0, 18), ("Coronal", 1, 15), ("Coronal", 0, 10), ("Axial", -1, 15)]
GEOM = dict(IMG=336, CROP_MM=140.0, SPAN=(0.02, 0.98))
OUT = Path("/kaggle/working") if IS_KAGGLE else Path("../experiments")
SPLIT = "train" if IS_KAGGLE else "test"   # local dry run: the test study on disk

spec = importlib.util.spec_from_file_location("rk", str(find_input("third_party/raptor/infer_knee.py") or "../third_party/raptor/infer_knee.py"))
rk = importlib.util.module_from_spec(spec); spec.loader.exec_module(rk)
rk.SLOTS, rk.MAXS = SLOTS80, 80
rk.IMG, rk.CROP_MM, rk.SPAN = GEOM["IMG"], GEOM["CROP_MM"], GEOM["SPAN"]

ser = pd.read_csv(DATA_DIR / f"{SPLIT}_series.csv").astype({"StudyInstanceUID": str, "SeriesInstanceUID": str})
SER = {k: v.to_dict("records") for k, v in ser.groupby("StudyInstanceUID")}
all_ids = sorted(SER)
if not IS_KAGGLE:
    all_ids = [s for s in all_ids if (DATA_DIR / "test_series" / s).exists()]
ids = list(np.array_split(np.array(all_ids), N_SHARDS)[SHARD]) if IS_KAGGLE else all_ids
TSDIR = str(DATA_DIR / f"{SPLIT}_series")
print(f"shard {SHARD}/{N_SHARDS}: {len(ids)} studies | {TSDIR}")

## Build the corpus (4 parallel processes, written straight into an on-disk memmap)

In [ ]:
_reader = None
def work(i_sid):
    global _reader
    if _reader is None:
        _reader = rk._make_reader()
    i, sid = i_sid
    try:
        v, m = rk.build_study(sid, SER, TSDIR, _reader)
        return i, v, m, ""
    except Exception as e:
        return i, None, None, f"{type(e).__name__}: {e}"

vols = np.lib.format.open_memmap(OUT / f"corpus_vols_s{SHARD:02d}.npy", mode="w+", dtype=np.uint8,
                                 shape=(len(ids), 80, GEOM["IMG"], GEOM["IMG"]))
masks = np.zeros((len(ids), 80), np.uint8)
errors, t0 = [], time.time()
pool = Pool(4) if IS_KAGGLE else None   # Pool does not work with notebook-defined functions on Windows
results = pool.imap_unordered(work, list(enumerate(ids)), chunksize=4) if pool else map(work, enumerate(ids))
if True:
    for n, (i, v, m, err) in enumerate(results):
        if err:
            errors.append((ids[i], err))
        else:
            vols[i] = v; masks[i] = m
        if (n + 1) % 100 == 0:
            print(f"  {n + 1}/{len(ids)}  {time.time() - t0:.0f}s", flush=True)
if pool:
    pool.close(); pool.join()
vols.flush(); del vols
np.save(OUT / f"corpus_masks_s{SHARD:02d}.npy", masks)
np.save(OUT / f"corpus_ids_s{SHARD:02d}.npy", np.array(ids, dtype=object), allow_pickle=True)
print(f"Bitti: {len(ids)} vaka, {(time.time() - t0) / 60:.1f} dk, hata {len(errors)}")

## Kalite kontrol

In [ ]:
valid = masks.sum(1)
print("valid slices per study:", pd.Series(valid).describe().round(1).to_dict())
print("slot fill rate:", {name: float((masks[:, a:b].sum(1) > 0).mean().round(3))
      for name, (a, b) in zip(["sag_fs", "sag_t1", "cor_fs", "cor_t1", "ax"], [(0, 22), (22, 40), (40, 55), (55, 65), (65, 80)])})
print("hatalar:", errors[:10])
assert len(errors) <= 0.01 * len(ids), "too many unreadable studies"